# Claim-Level Hallucination Rate

Addresses Examiner 2's comment: *"Apakah faktual consistency satu-satunya pengukur halusinasi? Sebaiknya ada pengukuran yang lebih valid — biasanya kita menghitung rasio, jumlah halusinasi / total claim dari sebuah label."*

The document-level `entailment_score` used everywhere else in this project (Table IV/V) is a single aggregate number per summary. This notebook adds a **claim-level** metric: for each summary, decompose it into individual claims and report what fraction of those claims are *not* supported by the source document.

Two complementary decompositions are used, both computed on the already-generated `predictions_baseline.jsonl` / `predictions_nli.jsonl` (no re-generation of summaries needed):

1. **Sentence-level NLI claims** — each sentence of the summary is treated as one claim, scored against the full source document via the same NLI model used throughout the project (`entailment` vs `neutral`/`contradiction`, i.e. the sentence-level `entailment_classifier` approach of Kryscinski et al. 2020 / FactCC, already cited in the thesis Bab II subbab 2.1.5). `hallucination_rate = (# non-entailed sentences) / (# sentences)` per summary, averaged over the dataset.
2. **Numeric claims** — a cheap, non-NLI sanity check: numbers/quantities mentioned in the summary are checked for literal presence in the source document (motivated directly by the qualitative "wrong number" failure case in thesis subbab 4.4.5, e.g. the 550 ribu / 490 ribu / 1.500 ribu case).

**Known limitation (state this honestly if asked):** the sentence splitter below is a regex heuristic, not a linguistic parser; it can occasionally over- or under-split on abbreviations or unusual punctuation. Given Liputan6 summaries are short (avg. 27 words, Table 3.2), most summaries decompose into 1–2 claims, which keeps this risk small but not zero.

## 1. Setup

In [ ]:
!pip install -q transformers

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## 2. Load Predictions

In [ ]:
import json
import glob
from pathlib import Path

# Flat dataset attached via Add Input, containing the two .jsonl files directly.
# Kaggle sometimes mounts a dataset under a slightly different slug than its display
# name (e.g. "prediction_21092026" -> "prediction-21092026" or with a numeric suffix),
# so if the guess below is wrong, a recursive glob under /kaggle/input finds it instead.
PREDICTIONS_BASELINE_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_baseline.jsonl"
PREDICTIONS_NLI_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_nli.jsonl"

if not Path(PREDICTIONS_BASELINE_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_baseline.jsonl", recursive=True)
    if hits:
        PREDICTIONS_BASELINE_FILE = hits[0]
if not Path(PREDICTIONS_NLI_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_nli.jsonl", recursive=True)
    if hits:
        PREDICTIONS_NLI_FILE = hits[0]

print("Baseline file:", PREDICTIONS_BASELINE_FILE, "| exists:", Path(PREDICTIONS_BASELINE_FILE).exists())
print("NLI file:", PREDICTIONS_NLI_FILE, "| exists:", Path(PREDICTIONS_NLI_FILE).exists())

if not Path(PREDICTIONS_BASELINE_FILE).exists() or not Path(PREDICTIONS_NLI_FILE).exists():
    print("\nNot found even after searching. Contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        print(" ", p)
    raise FileNotFoundError(
        "predictions_baseline.jsonl / predictions_nli.jsonl not found. Check the listing "
        "printed above for where the dataset actually landed, and that it is attached to "
        "THIS notebook via 'Add Input' in the right sidebar."
    )

# Set to an integer (e.g. 2000) to run on a random sample instead of the full 10,971 test
# documents — useful for a quick sanity check before committing to the full run.
SAMPLE_SIZE = None
SEED = 42

def load_rows(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

baseline_rows = load_rows(PREDICTIONS_BASELINE_FILE)
nli_rows = load_rows(PREDICTIONS_NLI_FILE)
print(f"Baseline predictions: {len(baseline_rows)}")
print(f"NLI predictions:      {len(nli_rows)}")

if SAMPLE_SIZE is not None:
    import random
    random.seed(SEED)
    idx = set(random.sample(range(len(baseline_rows)), min(SAMPLE_SIZE, len(baseline_rows))))
    baseline_rows = [r for i, r in enumerate(baseline_rows) if i in idx]
    nli_rows_by_id = {r["id"]: r for r in nli_rows}
    nli_rows = [nli_rows_by_id[r["id"]] for r in baseline_rows if r["id"] in nli_rows_by_id]
    print(f"Sampled down to {len(baseline_rows)} / {len(nli_rows)} paired rows (SAMPLE_SIZE={SAMPLE_SIZE})")

## 3. Load NLI Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32 if DEVICE == "cuda" else 8

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)
neu_idx = next(i for i, l in id2label.items() if "neutral" in l)
print(f"NLI model loaded on {DEVICE} (batch size {BATCH_SIZE})")

@torch.inference_mode()
def nli_predict_batch(premises, hypotheses):
    """Returns argmax label ('entailment'/'neutral'/'contradiction') per pair."""
    enc = nli_tokenizer(premises, hypotheses, truncation=True, max_length=512, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    probs = torch.softmax(nli_model(**enc).logits, dim=-1)
    preds = torch.argmax(probs, dim=-1).tolist()
    label_map = {ent_idx: "entailment", con_idx: "contradiction", neu_idx: "neutral"}
    return [label_map[p] for p in preds]

## 4. Sentence Splitter (Claim Decomposition)

In [ ]:
import re

def split_into_claims(summary: str):
    """Heuristic sentence splitter used as the claim-decomposition unit.
    Protects digit.digit patterns (e.g. "27.000", "3.5") from being treated as sentence
    boundaries, then splits on '.', '!', '?' followed by whitespace. Falls back to the
    whole summary as a single claim if no split point is found (common given the short,
    often single-sentence Liputan6 summaries — Table 3.2 reports an average of 27 words).
    """
    protected = re.sub(r"(?<=\d)\.(?=\d)", "§", summary)
    parts = re.split(r"(?<=[.!?])\s+", protected)
    parts = [p.replace("§", ".").strip() for p in parts if p.strip()]
    return parts if parts else [summary.strip()]

# Quick sanity check on a few examples
for ex in [
    "Pemerintah memberikan tenggat 14 hari. Mereka menolak kenaikan tarif dasar listrik.",
    "Sasaran penjualan aset BPPN sebesar Rp 27.000 triliun dinilai tak optimal.",
    "Kalimat tunggal tanpa titik di akhir",
]:
    print(split_into_claims(ex))

## 5. Sentence-Level Claim Hallucination Rate

In [ ]:
import statistics

def sentence_level_hallucination_rate(rows, label):
    per_summary_rates = []
    total_claims = 0
    total_non_entailed = 0
    n_batches_done = 0

    # Flatten all (premise, hypothesis, row_index) pairs across the whole dataset first,
    # so we can batch NLI calls efficiently instead of one tiny batch per summary.
    flat_premises, flat_hypotheses, flat_row_idx = [], [], []
    claims_per_row = []
    for row in rows:
        claims = split_into_claims(row["generated_summary"])
        claims_per_row.append(claims)
        for c in claims:
            flat_premises.append(row["document"])
            flat_hypotheses.append(c)
            flat_row_idx.append(len(claims_per_row) - 1)

    print(f"[{label}] {len(rows)} summaries -> {len(flat_premises)} sentence-level claims "
          f"(avg {len(flat_premises)/len(rows):.2f} claims/summary)")

    all_preds = [None] * len(flat_premises)
    for i in range(0, len(flat_premises), BATCH_SIZE):
        batch_p = flat_premises[i:i + BATCH_SIZE]
        batch_h = flat_hypotheses[i:i + BATCH_SIZE]
        preds = nli_predict_batch(batch_p, batch_h)
        all_preds[i:i + len(preds)] = preds
        n_batches_done += 1
        if n_batches_done % 50 == 0:
            print(f"  [{label}] scored {i + len(preds)}/{len(flat_premises)} claims", flush=True)

    # Aggregate back per-row
    non_entailed_per_row = [0] * len(claims_per_row)
    for pred, row_idx in zip(all_preds, flat_row_idx):
        if pred != "entailment":
            non_entailed_per_row[row_idx] += 1
            total_non_entailed += 1
    total_claims = len(flat_premises)

    for claims, n_bad in zip(claims_per_row, non_entailed_per_row):
        per_summary_rates.append(n_bad / len(claims))

    return {
        "label": label,
        "n_summaries": len(rows),
        "total_claims": total_claims,
        "total_non_entailed_claims": total_non_entailed,
        "claim_level_hallucination_rate": total_non_entailed / total_claims,
        "mean_per_summary_hallucination_rate": statistics.mean(per_summary_rates),
        "pct_summaries_with_any_hallucinated_claim": sum(1 for r in per_summary_rates if r > 0) / len(per_summary_rates),
    }

sentence_result_baseline = sentence_level_hallucination_rate(baseline_rows, "baseline")
sentence_result_nli = sentence_level_hallucination_rate(nli_rows, "nli")

print("\n=== Sentence-level claim hallucination rate ===")
for r in [sentence_result_baseline, sentence_result_nli]:
    print(f"{r['label']:10s} | claim-level rate = {r['claim_level_hallucination_rate']:.4f} "
          f"({r['total_non_entailed_claims']}/{r['total_claims']} claims) | "
          f"mean per-summary rate = {r['mean_per_summary_hallucination_rate']:.4f} | "
          f"% summaries with ≥ 1 hallucinated claim = {r['pct_summaries_with_any_hallucinated_claim']:.4f}")

## 6. Numeric-Claim Hallucination Rate (Complementary, Non-NLI Check)

In [ ]:
NUMBER_RE = re.compile(r"\d[\d.,]*\d|\d")

def extract_numbers(text, min_digits=2):
    normalized = []
    for raw in NUMBER_RE.findall(text):
        digits = re.sub(r"[.,]", "", raw)
        if len(digits) >= min_digits:
            normalized.append(digits)
    return normalized

def numeric_claim_hallucination_rate(rows, label):
    summaries_with_numbers = 0
    total_numbers = 0
    total_unsupported = 0
    per_summary_rates = []

    for row in rows:
        nums = extract_numbers(row["generated_summary"])
        if not nums:
            continue
        summaries_with_numbers += 1
        doc_digit_seqs = set(re.sub(r"[.,]", "", n) for n in NUMBER_RE.findall(row["document"]) if len(re.sub(r"[.,]", "", n)) >= 2)
        unsupported = sum(1 for n in nums if n not in doc_digit_seqs)
        total_numbers += len(nums)
        total_unsupported += unsupported
        per_summary_rates.append(unsupported / len(nums))

    if summaries_with_numbers == 0:
        return {"label": label, "summaries_with_numeric_claims": 0}

    return {
        "label": label,
        "n_summaries": len(rows),
        "summaries_with_numeric_claims": summaries_with_numbers,
        "pct_summaries_with_numeric_claims": summaries_with_numbers / len(rows),
        "total_numeric_claims": total_numbers,
        "total_unsupported_numeric_claims": total_unsupported,
        "numeric_claim_hallucination_rate": total_unsupported / total_numbers,
        "mean_per_summary_numeric_hallucination_rate": statistics.mean(per_summary_rates),
    }

numeric_result_baseline = numeric_claim_hallucination_rate(baseline_rows, "baseline")
numeric_result_nli = numeric_claim_hallucination_rate(nli_rows, "nli")

print("\n=== Numeric claim hallucination rate (regex-based, non-NLI sanity check) ===")
for r in [numeric_result_baseline, numeric_result_nli]:
    if r.get("summaries_with_numeric_claims", 0) == 0:
        print(f"{r['label']:10s} | no numeric claims found")
        continue
    print(f"{r['label']:10s} | {r['pct_summaries_with_numeric_claims']:.4f} of summaries contain a number | "
          f"claim-level rate = {r['numeric_claim_hallucination_rate']:.4f} "
          f"({r['total_unsupported_numeric_claims']}/{r['total_numeric_claims']} numbers) | "
          f"mean per-summary rate = {r['mean_per_summary_numeric_hallucination_rate']:.4f}")

## 7. Save Results

In [ ]:
output_path = Path("./results/claim_level_hallucination_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "sample_size": SAMPLE_SIZE,
    "n_paired_samples": len(baseline_rows),
    "sentence_level": {"baseline": sentence_result_baseline, "nli": sentence_result_nli},
    "numeric_level": {"baseline": numeric_result_baseline, "nli": numeric_result_nli},
}
with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print(f"Saved to {output_path}")

print("\n=== Copy into thesis Tabel 4.5 (new columns) ===")
print(f"{'Model':<12} | {'Claim-level rate (sentence)':>28} | {'% ringkasan >=1 klaim halusinasi':>32} | {'Claim-level rate (angka)':>26}")
print("-" * 105)
for s_key, n_key, name in [("baseline", "baseline", "BART Baseline"), ("nli", "nli", "BART + NLI")]:
    s = payload["sentence_level"][s_key]
    n = payload["numeric_level"][n_key]
    n_rate = n.get("numeric_claim_hallucination_rate", float("nan"))
    print(f"{name:<12} | {s['claim_level_hallucination_rate']:>28.4f} | "
          f"{s['pct_summaries_with_any_hallucinated_claim']:>32.4f} | {n_rate:>26.4f}")

## Narasi untuk disalin ke tesis (draf, sesuaikan dengan angka aktual hasil run)

Contoh kalimat untuk subbab 4.4.1 / 4.4.5, mengisi Examiner 2 #1:

> Selain skor *entailment* berbasis dokumen, penelitian ini juga menghitung metrik pelengkap berbasis klaim (*claim-level hallucination rate*), yaitu rasio jumlah klaim pada ringkasan yang tidak didukung dokumen sumber terhadap total klaim pada ringkasan tersebut. Klaim didekomposisi pada dua level: (a) level kalimat, mengikuti pendekatan *entailment classifier* sentence-level milik Kryscinski et al. (2020), dan (b) level angka/kuantitas, memverifikasi kemunculan literal angka pada dokumen sumber — termotivasi langsung oleh temuan kualitatif kesalahan angka pada subbab 4.4.5. Model baseline menghasilkan *claim-level hallucination rate* sebesar **[ISI DARI HASIL RUN]**, sedangkan model dengan integrasi NLI menurunkannya menjadi **[ISI DARI HASIL RUN]**, konsisten dengan arah perbaikan yang ditunjukkan oleh skor *entailment* berbasis dokumen pada Tabel 4.4/4.5.

Isi bagian `[ISI DARI HASIL RUN]` dengan angka `claim_level_hallucination_rate` dari cell output di atas setelah notebook ini selesai dijalankan.